# 01. 이미지 폴더를 Label Studio project에 import

이 노트북은 이미지 폴더를 Label Studio task로 등록합니다.

기본값은 `DRY_RUN=True`입니다. 이 상태에서는 Label Studio에 아무 것도 만들지 않고, 어떤 `/data/local-files/?d=...` URL이 만들어질지만 확인합니다.

사용 순서:

1. `.env`에 `LABEL_STUDIO_URL`, `LABEL_STUDIO_API_KEY`, `LABEL_STUDIO_DOC_ROOT`를 설정합니다.
2. 아래 설정 cell에서 `SRC_ROOT`를 import할 이미지 폴더로 바꿉니다.
3. 새 project를 만들려면 `PROJECT_ID = None`으로 둡니다.
4. 기존 project에 추가하려면 `PROJECT_ID = 숫자`로 바꿉니다.
5. 먼저 `DRY_RUN=True`로 URL이 맞는지 확인합니다.
6. 문제가 없으면 `DRY_RUN=False`로 바꾸고 실행합니다.

`mirror_root`는 `SRC_ROOT`가 `doc_root` 밖에 있을 때만 필요합니다. 자세한 설명은 `docs/path-and-mount-guide.md`를 확인하세요.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src").is_dir():
            return path
    raise RuntimeError("repo root를 찾지 못했습니다.")

REPO_ROOT = find_repo_root(Path.cwd())
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from labelstudio_bbox_tools.config import settings_from_env
from labelstudio_bbox_tools.importers.image_import import import_images

In [ ]:
# ===== 사용자가 주로 바꾸는 설정 =====

# import할 이미지 폴더입니다. 서버마다 다르므로 본인 경로로 바꾸세요.
SRC_ROOT = Path("/path/to/your/images")

# None이면 새 Label Studio project를 만듭니다.
# 기존 project에 이미지를 추가하려면 예: PROJECT_ID = 123 처럼 숫자를 넣으세요.
PROJECT_ID = None

# 새 project를 만들 때 사용할 제목입니다.
# 실제 생성 후에는 코드가 자동으로 "project_id_제목" 형식으로 바꿉니다.
PROJECT_TITLE = "my_bbox_import_test"

# 하위 폴더까지 이미지 검색하려면 True.
RECURSIVE = False

# 처음 테스트할 때는 일부만 import하는 것이 안전합니다.
# 예: ":10"은 앞에서 10장만, None은 전체입니다.
SLICE_SPEC = ":5"

# SRC_ROOT가 LABEL_STUDIO_DOC_ROOT 밖에 있으면 mirror root를 지정하세요.
# 예: MIRROR_ROOT = Path("/data/datasets")
MIRROR_ROOT = None

# True면 실제 project/task를 만들지 않습니다. 처음에는 반드시 True로 확인하세요.
DRY_RUN = True

In [ ]:
settings = settings_from_env(REPO_ROOT / ".env")
print("Label Studio URL:", settings.url)
print("DOC_ROOT:", settings.doc_root)
print("SRC_ROOT:", SRC_ROOT)
print("DRY_RUN:", DRY_RUN)

In [ ]:
result = import_images(
    src_root=SRC_ROOT,
    ls_url=settings.url,
    api_key=settings.api_key,
    doc_root=settings.doc_root,
    recursive=RECURSIVE,
    slice_spec=SLICE_SPEC,
    project_id=PROJECT_ID,
    project_title=PROJECT_TITLE,
    mirror_root=MIRROR_ROOT,
    dry_run=DRY_RUN,
)

print(result)